# Requirements

In [49]:
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


# Import of packages

In [114]:
import io
import re
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
import pdfplumber
import requests
import time
from bs4 import BeautifulSoup

In [115]:
class Params:
    BASE_URL = "https://www.jpx.co.jp"
    PAGE_URL = BASE_URL + "/english/listing/market-alerts/supervision/index.html"
    TIMEOUT = 30
    MAX_WORKERS = 12

In [99]:
class JpxDelisted:
    DATE = r"([A-Z][a-z]+\.?\s*\d{1,2},\s*\d{4})"
    PATTERNS = [
        re.compile(r"Date\s+of\s+Delisting\s*[:：]?\s*" + DATE, re.I),
        re.compile(r"Delisting\s+Date\s*[:：]?\s*" + DATE, re.I),
        re.compile(r"delist\w*\D{0,60}?" + DATE, re.I),
    ]
    def __init__(self):
        self.session = requests.Session()

    def fetch_page(self):
        response = self.session.get(Params.PAGE_URL, timeout=Params.TIMEOUT)
        response.raise_for_status()
        return BeautifulSoup(response.text, "lxml")

    def find_delisted_table(self, soup):
        header = soup.find(
            lambda tag: tag.name in ("h2", "h3", "h4")
            and "securities to be delisted" in tag.get_text(strip=True).lower()
        )
        table = header.find_next("table")
        rows = []
        for tr in table.find_all("tr"):
            cells = tr.find_all("td")
            if len(cells) < 5:
                continue
            link = cells[4].find("a", href=True)
            pdf_url = Params.BASE_URL + link["href"] if link else ""
            rows.append({
                "designation_date": cells[0].get_text(strip=True),
                "name": cells[1].get_text(strip=True),
                "code": cells[2].get_text(strip=True),
                "market_segment": cells[3].get_text(strip=True),
                "pdf_url": pdf_url,
                "delisting_date": None,
            })
        return rows

    def delisting_date(self, pdf_url):
        if not pdf_url:
            return None
        try:
            resp = self.session.get(pdf_url, timeout=Params.TIMEOUT)
            resp.raise_for_status()
            with pdfplumber.open(io.BytesIO(resp.content)) as pdf:
                text = "\n".join(page.extract_text() or "" for page in pdf.pages)
        except Exception as exc:
            print(f"[WARN] {pdf_url}: {exc}")
            return None
        text = re.sub(r"\s+", " ", text)
        for pattern in self.PATTERNS:
            m = pattern.search(text)
            if m:
                return m.group(1).strip()
        return None

    def batch(self, rows, max_workers=Params.MAX_WORKERS,multi_threading=True):
        if multi_threading:
            with ThreadPoolExecutor(max_workers=max_workers) as pool:
                dates = pool.map(self.delisting_date, [r["pdf_url"] for r in rows])
            for r, d in zip(rows, dates):
                r["delisting_date"] = d
        else:
            for r in rows:
                r["delisting_date"] = self.delisting_date(r["pdf_url"])
        return rows

    def dataframe(self, rows):
        df = pd.DataFrame(rows)
        df["delisting_date_dt"] = pd.to_datetime(
            df["delisting_date"].str.replace(".", "", regex=False), errors="coerce"
        )
        df = df.sort_values("delisting_date_dt").reset_index(drop=True)
        df.to_excel("Securities to be delisted.xlsx", index=False)
        return df

    def benchmark(self):
        soup = self.fetch_page()
        rows_seq = self.find_delisted_table(soup)
        t0 = time.perf_counter()
        self.batch(rows_seq, multi_threading=False)
        t_seq = time.perf_counter() - t0
    
        rows_par = self.find_delisted_table(soup)
        t0 = time.perf_counter()
        self.batch(rows_par, multi_threading=True)
        t_par = time.perf_counter() - t0
    
        return pd.DataFrame({
            "mode": ["séquentiel", f"multithreadé ({Params.MAX_WORKERS} workers)"],
            "pdfs": [len(rows_seq)] * 2,
            "temps (s)": [round(t_seq, 2), round(t_par, 2)],
            "speedup": [1.0, round(t_seq / t_par, 1)],
        })

    def evaluate(self):
        soup = self.fetch_page()
        rows = self.find_delisted_table(soup)
        rows = self.batch(rows)
        return self.dataframe(rows)

# Results

In [100]:
JpxDelisted = JpxDelisted()
df = JpxDelisted.evaluate()
print(df)

   designation_date                                        name  code  \
0     Jul. 01, 2026                             SHINPO CO.,LTD.  5903   
1     Jul. 09, 2026                             The Global Ltd.  3271   
2     Jul. 06, 2026                         Solasto Corporation  6197   
3     Jul. 15, 2026                          PALTAC CORPORATION  8283   
4     Jul. 13, 2026                     Global Information,Inc.  4171   
5      May 13, 2026                     Uematsu Shokai Co.,Ltd.  9914   
6     Apr. 15, 2026                                KOYOSHA INC.  7946   
7     Apr. 15, 2026                                  NEPON Inc.  7985   
8     Apr. 22, 2026                         Yamadai Corporation  7426   
9     Apr. 22, 2026                         KUBOTEK CORPORATION  7709   
10     May 13, 2026           KAWASE COMPUTER SUPPLIES CO.,LTD.  7851   
11    Apr. 15, 2026                                SAKURAI LTD.  7255   
12     May 27, 2026                           YAMAZ

# What if we didn't use multi threading ? 

In [65]:
JpxDelisted.benchmark()

,mode,pdfs,temps (s),speedup
0,séquentiel,17,7.52,1.0
1,multithreadé (12 workers),17,0.72,10.4
